In [11]:
# ============================================================================
# EMIPredict AI - Fast Leakage-Proof ML Pipeline with MLflow
# ============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, PowerTransformer, QuantileTransformer, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTENC
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from scipy.stats import randint, uniform

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow import MlflowClient

from classification_feature_engineering import ClassificationFeatureEngineer


In [12]:
mlflow.set_tracking_uri("http://13.204.193.251:5000")
mlflow.set_experiment("EMI_Classification_Experiment")

<Experiment: artifact_location='s3://mlflow-tracking-loan/5', creation_time=1764526983582, experiment_id='5', last_update_time=1764526983582, lifecycle_stage='active', name='EMI_Classification_Experiment', tags={}>

In [13]:

df = pd.read_csv(r"D:\backup\AI course work\Guvi\Assignement_2\data\clean_emi_data.csv")
df.dropna(subset=["emi_eligibility"], inplace=True)

X = df.drop(["emi_eligibility", "max_monthly_emi"], axis=1)
y = df["emi_eligibility"]

print(f"Data: {df.shape}, Target: {y.value_counts().to_dict()}\n")

Data: (404800, 27), Target: {'Not_Eligible': 312868, 'Eligible': 74444, 'High_Risk': 17488}



In [14]:
# ======================================
# TRAIN-TEST SPLIT
# ======================================
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (323840, 25), Test: (80960, 25)


In [15]:
# ======================================
# APPLY SMOTENC (LEAKAGE-PROOF)
# ======================================
categorical_cols = ["gender", "marital_status", "education", "employment_type",
                   "company_type", "house_type", "existing_loans", "emi_scenario"]

categorical_indices = [X_train.columns.get_loc(c) for c in categorical_cols]
smote = SMOTENC(categorical_features=categorical_indices, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"After SMOTE: {X_train_balanced.shape}\n")

After SMOTE: (750882, 25)



In [16]:
# ======================================
# PREPROCESSING
# ======================================
nominal_cols = ["gender", "marital_status", "employment_type", "company_type", "house_type", "emi_scenario"]
ordinal_cols = ["education"]
binary_cols = ["existing_loans"]
education_order = ["High School", "Graduate", "Professional", "Post Graduate"]

# Determine skewness
fe_temp = ClassificationFeatureEngineer()
tmp = fe_temp.fit_transform(X_train_balanced)
num_cols = tmp.select_dtypes(include=["int64", "float64"]).columns
sk = tmp[num_cols].skew()

low_skew = sk[abs(sk) <= 0.5].index.tolist()
mid_skew = sk[(abs(sk) > 0.5) & (abs(sk) <= 1)].index.tolist()
high_skew = sk[abs(sk) > 1].index.tolist()

preprocessor = ColumnTransformer([
    ("low", Pipeline([("impute", SimpleImputer(strategy="median")), 
                      ("scale", StandardScaler())]), low_skew),
    ("mid", Pipeline([("impute", SimpleImputer(strategy="median")), 
                      ("power", PowerTransformer()), 
                      ("scale", StandardScaler())]), mid_skew),
    ("high", Pipeline([("impute", SimpleImputer(strategy="median")), 
                       ("quantile", QuantileTransformer(output_distribution="normal")), 
                       ("scale", StandardScaler())]), high_skew),
    ("nominal", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))]), nominal_cols),
    ("ordinal", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("ord", OrdinalEncoder(categories=[education_order], handle_unknown='use_encoded_value', unknown_value=-1))]), ordinal_cols),
    ("binary", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                         ("ord", OrdinalEncoder(categories=[["No", "Yes"]], handle_unknown='use_encoded_value', unknown_value=-1))]), binary_cols)
], verbose_feature_names_out=False)

In [17]:
# ======================================
# BUILD PIPELINE
# ======================================
def build_pipeline(model):
    selector = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), threshold='median')
    return Pipeline([
        ("feature_eng", ClassificationFeatureEngineer()),
        ("preprocess", preprocessor),
        ("select", selector),
        ("model", model)
    ])


In [8]:
# ======================================
# BASELINE EVALUATION
# ======================================
print("="*60)
print("BASELINE MODEL EVALUATION")
print("="*60)

models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random_Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1, verbosity=0),
    "Decision_Tree": DecisionTreeClassifier(random_state=42),
    "Gradient_Boosting": GradientBoostingClassifier(random_state=42)
}

baseline_results = {}

for name, model in models.items():
    print(f"\n{name}...")
    
    with mlflow.start_run(run_name=f"Baseline_{name}"):
        mlflow.log_param("model_type", name)
        
        pipe = build_pipeline(model)
        pipe.fit(X_train_balanced, y_train_balanced)
        
        y_test_pred = pipe.predict(X_test)
        
        try:
            y_test_proba = pipe.predict_proba(X_test)
            if len(le.classes_) == 2:
                roc_auc = roc_auc_score(y_test, y_test_proba[:, 1])
            else:
                roc_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='weighted')
        except:
            roc_auc = None
        
        acc = accuracy_score(y_test, y_test_pred)
        f1 = f1_score(y_test, y_test_pred, average='weighted')
        precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
        
        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1", f1)
        mlflow.log_metric("test_precision", precision)
        mlflow.log_metric("test_recall", recall)
        if roc_auc is not None:
            mlflow.log_metric("test_roc_auc", roc_auc)
        
        cm = confusion_matrix(y_test, y_test_pred)
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
        plt.title(f'{name}')
        plt.ylabel('True')
        plt.xlabel('Predicted')
        mlflow.log_figure(fig, f"cm_{name}.png")
        plt.close()
        
        # Log dataset info as artifact (only once)
        if name == list(models.keys())[0]:
            dataset_info = {
                'train_shape': X_train_balanced.shape,
                'test_shape': X_test.shape,
                'target_classes': le.classes_.tolist(),
                'features': X_train.columns.tolist()
            }
            with open("dataset_info.txt", "w") as f:
                for key, value in dataset_info.items():
                    f.write(f"{key}: {value}\n")
            mlflow.log_artifact("dataset_info.txt")
        
        baseline_results[name] = {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall, 'roc_auc': roc_auc if roc_auc else 0}
        
        print(f"  Accuracy: {acc:.4f}, F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, ROC-AUC: {roc_auc:.4f}" if roc_auc else f"  Accuracy: {acc:.4f}, F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

results_df = pd.DataFrame(baseline_results).T.sort_values('f1', ascending=False)
print("\n" + "="*60)
print("BASELINE RESULTS")
print("="*60)
print(results_df)

BASELINE MODEL EVALUATION

Logistic_Regression...
  Accuracy: 0.8760, F1: 0.9043, Precision: 0.9531, Recall: 0.8760, ROC-AUC: 0.9821
🏃 View run Baseline_Logistic_Regression at: http://13.204.193.251:5000/#/experiments/5/runs/081e9ad8c9aa443d90e580a756ae4069
🧪 View experiment at: http://13.204.193.251:5000/#/experiments/5

Random_Forest...
  Accuracy: 0.9446, F1: 0.9503, Precision: 0.9603, Recall: 0.9446, ROC-AUC: 0.9946
🏃 View run Baseline_Random_Forest at: http://13.204.193.251:5000/#/experiments/5/runs/da306f6460674fb4aa10abc15a835f02
🧪 View experiment at: http://13.204.193.251:5000/#/experiments/5

XGBoost...
  Accuracy: 0.9482, F1: 0.9536, Precision: 0.9634, Recall: 0.9482, ROC-AUC: 0.9953
🏃 View run Baseline_XGBoost at: http://13.204.193.251:5000/#/experiments/5/runs/1c1019ba63cf458299332436aafa85dc
🧪 View experiment at: http://13.204.193.251:5000/#/experiments/5

Decision_Tree...
  Accuracy: 0.9266, F1: 0.9340, Precision: 0.9456, Recall: 0.9266, ROC-AUC: 0.9399
🏃 View run Baselin

In [9]:
best_model_name = results_df.index[0]
print(f"\n✓ Best Model: {best_model_name}\n")


✓ Best Model: XGBoost



In [18]:
best_model_name = "XGBoost"

In [19]:
# ==========================================================
# REPLACEMENT BLOCK — NO HYPERPARAMETER TUNING
# USING CROSS-VALIDATION FOR FINAL MODEL EVALUATION
# ==========================================================

print("\n" + "="*60)
print(f"FINAL MODEL SELECTION (NO TUNING REQUIRED) → {best_model_name}")
print("="*60)

final_model = models[best_model_name]
final_pipe = build_pipeline(final_model)

# ---------- 1. CROSS-VALIDATION ----------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_validate(
    final_pipe,
    X_train_balanced,
    y_train_balanced,
    cv=cv,
    scoring=[
        'accuracy',
        'f1_weighted',
        'precision_weighted',
        'recall_weighted',
        'roc_auc_ovr_weighted'
    ],
    n_jobs=-1,
    return_train_score=False
)

print("\nCROSS-VALIDATION PERFORMANCE (5-Fold):")
print(f"Accuracy:      {cv_scores['test_accuracy'].mean():.4f} ± {cv_scores['test_accuracy'].std():.4f}")
print(f"F1-Weighted:   {cv_scores['test_f1_weighted'].mean():.4f} ± {cv_scores['test_f1_weighted'].std():.4f}")
print(f"Precision:     {cv_scores['test_precision_weighted'].mean():.4f} ± {cv_scores['test_precision_weighted'].std():.4f}")
print(f"Recall:        {cv_scores['test_recall_weighted'].mean():.4f} ± {cv_scores['test_recall_weighted'].std():.4f}")
print(f"ROC-AUC:       {cv_scores['test_roc_auc_ovr_weighted'].mean():.4f} ± {cv_scores['test_roc_auc_ovr_weighted'].std():.4f}")

# ---------- 2. RETRAIN FINAL MODEL ON FULL TRAINING SET ----------
final_pipe.fit(X_train_balanced, y_train_balanced)

# ---------- 3. FINAL TEST EVALUATION ----------
y_test_pred = final_pipe.predict(X_test)
y_test_proba = final_pipe.predict_proba(X_test)

if len(le.classes_) == 2:
    roc_auc = roc_auc_score(y_test, y_test_proba[:, 1])
else:
    roc_auc = roc_auc_score(y_test, y_test_proba, multi_class="ovr", average="weighted")

test_acc = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')
test_precision = precision_score(y_test, y_test_pred, average='weighted')
test_recall = recall_score(y_test, y_test_pred, average='weighted')

print("\nFINAL TEST PERFORMANCE:")
print(f"Accuracy:      {test_acc:.4f}")
print(f"F1-Weighted:   {test_f1:.4f}")
print(f"Precision:     {test_precision:.4f}")
print(f"Recall:        {test_recall:.4f}")
print(f"ROC-AUC:       {roc_auc:.4f}")

# ---------- 4. MLflow Logging ----------
with mlflow.start_run(run_name=f"Final_{best_model_name}_No_Tuning"):

    mlflow.log_param("model_type", best_model_name)
    mlflow.log_param("hyperparameter_tuning", "Not Required — Strong Baseline")

    # CV metrics
    mlflow.log_metric("cv_accuracy_mean", cv_scores['test_accuracy'].mean())
    mlflow.log_metric("cv_f1_mean", cv_scores['test_f1_weighted'].mean())
    mlflow.log_metric("cv_precision_mean", cv_scores['test_precision_weighted'].mean())
    mlflow.log_metric("cv_recall_mean", cv_scores['test_recall_weighted'].mean())
    mlflow.log_metric("cv_roc_auc_mean", cv_scores['test_roc_auc_ovr_weighted'].mean())

    # Test metrics
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.log_metric("test_f1", test_f1)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("test_recall", test_recall)
    mlflow.log_metric("test_roc_auc", roc_auc)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_test_pred)
    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f"Final Model — {best_model_name}")
    mlflow.log_figure(fig, "final_confusion_matrix.png")
    plt.close()

    # Log model
    signature = infer_signature(X_train_balanced, final_pipe.predict(X_train_balanced))
    mlflow.sklearn.log_model(
        final_pipe,
        "model",
        signature=signature,
        registered_model_name=f"EMI_Classification_{best_model_name}"
    )

print("\n🎯 FINAL MODEL SAVED — NO TUNING — CV VALIDATED\n")



FINAL MODEL SELECTION (NO TUNING REQUIRED) → XGBoost

CROSS-VALIDATION PERFORMANCE (5-Fold):
Accuracy:      0.9442 ± 0.0004
F1-Weighted:   0.9444 ± 0.0004
Precision:     0.9451 ± 0.0004
Recall:        0.9442 ± 0.0004
ROC-AUC:       0.9924 ± 0.0001

FINAL TEST PERFORMANCE:
Accuracy:      0.9482
F1-Weighted:   0.9535
Precision:     0.9631
Recall:        0.9482
ROC-AUC:       0.9953


2025/12/01 10:52:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'EMI_Classification_XGBoost'.
2025/12/01 10:54:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: EMI_Classification_XGBoost, version 1
Created version '1' of model 'EMI_Classification_XGBoost'.


🏃 View run Final_XGBoost_No_Tuning at: http://13.204.193.251:5000/#/experiments/5/runs/32e2c991bb7f4d5a98cd35896314a504
🧪 View experiment at: http://13.204.193.251:5000/#/experiments/5

🎯 FINAL MODEL SAVED — NO TUNING — CV VALIDATED



In [20]:
# ---- Transition Model to Production ----
client = MlflowClient()

try:
    latest_version = client.get_latest_versions(
        f"EMI_Classification_{best_model_name}", stages=["None"]
    )[0].version

    client.transition_model_version_stage(
        name=f"EMI_Classification_{best_model_name}",
        version=latest_version,
        stage="Production"
    )

    print(f"\n✓ Model registered & moved to PRODUCTION → Version {latest_version}")

except Exception as e:
    print("\n⚠ Model registered but stage transition failed.")
    print(e)



✓ Model registered & moved to PRODUCTION → Version 1
